Air Traffic Control

Chromosome - list of the rules like the hight, distance, altitude of each plane flying in the air avoiding crash.

---
Basic Imports

In [43]:
import numpy as np
import pandas as pd
from copy import deepcopy
import array as arr
from datetime import datetime, timedelta
import random

In [ ]:
def closest_pass(p1, p2, v1, v2, time_window, minimum_distance):
    delta_p = p1 - p2
    delta_v = v1 - v2
    a = np.dot(delta_v, delta_v) # Speed difference between the planes
    b = 2 * np.dot(delta_p, delta_v)# Relative speed between the planes
    c = np.dot(delta_p, delta_p) # Distance between the planes

    print ("a:", a, "b:", b, "c:", c)

    if a == 0:
        return np.sqrt(c)
    t = -b / (2 * a)

    print("Time of closest approach:", t)
    if t < 0:
        return 0
    if t > time_window:
        return 0    
        
    closest_distance = np.sqrt(c + t * (b + a * t))
    
    print("Closest distance:", closest_distance)
    if closest_distance < minimum_distance:
        return 100
    return 0


def convert_long_lat_alt_to_cartesian(long, lat, alt):
    y=lat * 111111
    x = long * 111111 * np.cos(np.radians(lat))
    z = alt
    return np.array([x, y, z])


def get_cartesian_velocity(vertrate, speed, heading):
    heading_rad = np.radians(heading)

    v_z = speed * np.cos(heading_rad)
    v_x = speed * np.sin(heading_rad)
    v_y = vertrate 
    
    return np.array([v_x, v_y, v_z])

In [38]:
#plane 1
lat1 =49.5354448739
long1 =2.83373033678
speed1 =181.779353382
head_1 =66.4820383263
vertrate1 = 4.55168
alt1 = 4838.7

#plane 2
lat2 =51.7063293457
long2 =-3.28788138725
speed2 = 236.560348391
head_2 =316.762391024
vertrate2 = 0.32512
alt2 = 11841.48

#plane 3
lat3 =49.542388916
long3 =2.85810771741
speed3 = 181.985246682
head_3 =246.3335256222
vertrate3 = 5.20192
alt3 = 4892.04

In [39]:
pos1 = convert_long_lat_alt_to_cartesian(long1, lat1, alt1)
vel1 = get_cartesian_velocity(vertrate1, speed1, head_1)

pos2 = convert_long_lat_alt_to_cartesian(long2, lat2, alt2)
vel2 = get_cartesian_velocity(vertrate2, speed2, head_2) 

pos3 = convert_long_lat_alt_to_cartesian(long3, lat3, alt3)
vel3 = get_cartesian_velocity(vertrate3, speed3, head_3)

In [42]:
print("Plane 1 position:", pos1, "velocity:", vel1)
print("Plane 3 position:", pos3, "velocity:", vel3)

closest_pass(pos1, pos3, vel1, vel3, 60, 6000)

Plane 1 position: [2.04336159e+05 5.50393282e+06 4.83870000e+03] velocity: [166.679856   4.55168   72.536604]
Plane 3 position: [2.06064693e+05 5.50470437e+06 4.89204000e+03] velocity: [-166.679856    5.20192   -73.051048]
a: 132324.88481120148 b: -1166974.9949519383 c: 3585978.4761788216
Time of closest approach: 4.409506936722276
Closest distance: 1006.5218867803194


100

# TODO:

import the dataset 

get the convert convert_long_lat_alt_to_cartesian and get_cartesian_velocity for each plane

compare each plane to each other to get clothest pass

In [ ]:
flights_df = pd.read_csv("Data/flights_to_dublin.csv")
print(flights_df.head())

         time  icao24        lat       lon    velocity     heading  vertrate  \
0  1527548410  4ca1d2  49.535445  2.833730  181.779353   66.482038   4.55168   
1  1527548410  4ca5ea  51.706329 -3.287881  236.560348  316.762391   0.32512   
2  1527548410  4ca8d7  53.679060 -5.311567  162.050677  269.818109  -4.87680   
3  1527548410  4ca303  53.778488 -6.063354  123.908406  261.163409  -2.60096   
4  1527548410  4ca97c  50.252563 -3.978424  226.215014  338.101807   0.00000   

   callsign  onground  alert    spi  squawk  baroaltitude  geoaltitude  \
0  RYR52GH      False  False  False   674.0       4701.54      4838.70   
1  RYR17XT      False  False  False  5701.0      11582.40     11841.48   
2  RYR73UF      False  False  False  2522.0       3817.62      3992.88   
3  RYR39BN      False  False  False  2002.0       2141.22      2270.76   
4  RYR74UE      False  False  False  7474.0      11582.40     11833.86   

   lastposupdate   lastcontact  
0   1.527548e+09  1.527548e+09  
1   1.52

In [ ]:
class problem:
    def __init__(self, number_of_planes):
        self.number_of_planes = number_of_planes
        self.genes_per_plane = 5  
        self.number_of_genes = self.number_of_planes * self.genes_per_plane
        #self.cost_function = cost_function

In [6]:
class Parameters:
    def __init__(self):
        self.population = 400
        self.maximum_number_of_generations = 1000
        self.crossover_exploration = 0.5
        self.mutation_rate = 0.2
        self.mutation_range = 2.0
        self.child_factor = 1

In [7]:
class Individual:
    def __init__(self, prob):
        planes = generate_genes(prob.number_of_planes)
        print("Generated planes:")
        print(planes)

        self.chromosome = convert_genes_values(planes)
        print("Fixed chromosome:")
        print(self.chromosome)
        self.cost = prob.cost_function(self.chromosome)

    def Crossover(self, other, epsilon):
        child1 = deepcopy(self)
        child2 = deepcopy(other)

        alpha = np.random.uniform(-epsilon,1+epsilon)

        child1.chromosome = alpha*self.chromosome + (1-alpha)*other.chromosome
        child2.chromosome = (1-alpha)*self.chromosome + alpha*other.chromosome
        return child1, child2
    
    def Mutate(self,mutation_rate,mutation_range):
        for i in range(len(self.chromosome)):
            if np.random.uniform(0,1) < mutation_rate:
                self.chromosome[i] += np.random.randn()*mutation_range

In [ ]:
def get_parents(population):
    ##longic
    return population

In [ ]:
def run_genetic(prob, params):
    ##longic
    return prob

In [10]:
prob = problem(number_of_planes=10)
test_individual = Individual(prob)

Generated planes:
[['05', (4934.412283088523, -9677.333857321444), 53.90679109334262, 270.0637634043381, 'RW-3', 'East'], ['44', (-3490.685605379591, -6548.104001586721), 1226.4828762509144, 270.62274197178306, 'RW-1', 'West'], ['03', (-3450.1300318952844, -5040.053573515273), 557.476622561339, 207.0929302184234, 'RW-2', 'East'], ['11', (6720.313709097056, -6623.077326059987), 1267.3521657041429, 282.1703250690514, 'RW-2', 'West'], ['17', (4233.3363660091345, 3868.4840403108756), 179.31581759778982, 232.1786246106288, 'RW-1', 'West'], ['45', (-2851.6339424604475, -4090.6271370617396), 820.7643362395653, 223.24495328116734, 'RW-2', 'East'], ['53', (8126.674302175939, 1989.327548596546), 1548.1036089922354, 277.75439423012216, 'RW-1', 'East'], ['17', (-8874.475253585288, 6969.808923493041), 1044.316148723315, 248.3082121144067, 'RW-3', 'West'], ['43', (-6392.267990652705, -653.4820838272954), 129.96376245832764, 203.022038522773, 'RW-3', 'West'], ['27', (-5928.1713195018365, 800.47340349